# 📊 Aula 05 — ETL: Extract, Transform and Load

## 🔄 Do dado bruto ao dado preparado para análise

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Pandas

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender o conceito de ETL;
- Diferenciar **Extract, Transform e Load**;
- Extrair dados de diferentes fontes;
- Transformar e padronizar dados;
- Combinar diferentes conjuntos de dados;
- Criar uma base preparada para análise;
- Compreender a importância do ETL em projetos de Mineração de Dados;
- Aplicar um fluxo ETL ao projeto do motor elétrico;
- Identificar como o processo ETL poderá ser utilizado no próprio projeto.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. Seu projeto de avaliação deverá utilizar outro tema.


# 🏭 1. O problema dos dados espalhados

Imagine que a indústria possui informações sobre os motores em diferentes sistemas:

```text
Sistema de sensores
        ↓
CSV com medições

Sistema de manutenção
        ↓
CSV com informações dos motores

Sistema de produção
        ↓
CSV com horas de operação
```

Nenhum arquivo possui sozinho todas as informações necessárias.

Precisamos:

1. **Extrair** os dados;
2. **Transformar** os dados;
3. **Carregar** os dados em um local adequado para análise.

Esse processo é conhecido como **ETL**.


# 🔄 2. O que significa ETL?

## E — Extract (Extração)

Obter dados de uma ou mais fontes.

Exemplos:

- CSV;
- Excel;
- banco de dados;
- API;
- sistema corporativo;
- sensores;
- páginas da web.

---

## T — Transform (Transformação)

Modificar os dados para que possam ser utilizados.

Exemplos:

- corrigir tipos;
- tratar valores ausentes;
- padronizar textos;
- converter unidades;
- criar novas colunas;
- combinar tabelas;
- remover duplicidades.

---

## L — Load (Carga)

Salvar os dados transformados em um destino.

Exemplos:

- CSV;
- Excel;
- banco de dados;
- Data Warehouse;
- sistema de análise.

---

### Fluxo

```text
FONTES
  ↓
EXTRACT
  ↓
TRANSFORM
  ↓
LOAD
  ↓
BASE PRONTA PARA ANÁLISE
```


# 🏭 3. ETL no projeto do motor

Vamos imaginar três fontes.

### Fonte 1 — Sensores

```text
motor
data_hora
corrente
tensao
vibracao
temperatura
rpm
```

### Fonte 2 — Cadastro dos motores

```text
motor
modelo
fabricante
potencia_nominal
linha
```

### Fonte 3 — Manutenção

```text
motor
ultima_manutencao
horas_operacao
status_manutencao
```

Nosso objetivo será criar uma única base:

```text
motor
data_hora
corrente
tensao
vibracao
temperatura
rpm
modelo
fabricante
potencia_nominal
linha
ultima_manutencao
horas_operacao
status_manutencao
```

Essa base poderá posteriormente ser utilizada na análise e na Mineração de Dados.


# 💻 4. EXTRACT — Extraindo dados

Vamos simular três arquivos CSV.

Primeiro criaremos os DataFrames que representarão as fontes.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
sensores = pd.DataFrame({
    "motor": ["M001", "M001", "M002", "M002", "M003", "M003"],
    "data_hora": [
        "2026-02-01 08:00", "2026-02-01 09:00",
        "2026-02-01 08:00", "2026-02-01 09:00",
        "2026-02-01 08:00", "2026-02-01 09:00"
    ],
    "corrente": [12.4, 12.8, 10.8, 11.0, 14.1, 14.3],
    "tensao": [380, 379, 380, 381, 382, 381],
    "vibracao": [1.8, 2.1, 1.2, 1.3, 2.4, 2.6],
    "temperatura": [62.3, 64.1, 55.1, 56.0, 67.2, 68.4],
    "rpm": [1750, 1748, 1752, 1750, 1740, 1738]
})

sensores

In [ ]:
cadastro = pd.DataFrame({
    "motor": ["M001", "M002", "M003"],
    "modelo": ["MX-100", "MX-100", "MX-200"],
    "fabricante": ["MotorTech", "MotorTech", "PowerMotor"],
    "potencia_nominal": [7.5, 6.5, 9.0],
    "linha": ["Linha A", "Linha A", "Linha B"]
})

cadastro

In [ ]:
manutencao = pd.DataFrame({
    "motor": ["M001", "M002", "M003"],
    "ultima_manutencao": ["2026-01-15", "2026-01-20", "2025-12-18"],
    "horas_operacao": [1254, 824, 2104],
    "status_manutencao": ["Em dia", "Em dia", "Atenção"]
})

manutencao

## Extração de arquivos

Em uma situação real, poderíamos utilizar:

```python
pd.read_csv("sensores.csv")
```

ou:

```python
pd.read_excel("cadastro.xlsx")
```

Nesta aula estamos criando os dados diretamente no notebook para concentrar a atenção no processo ETL.


# 🔧 5. TRANSFORM — Padronizando os dados

A transformação começa pela inspeção das fontes.

Vamos verificar os tipos.


In [ ]:
sensores.dtypes

In [ ]:
cadastro.dtypes

In [ ]:
manutencao.dtypes

Datas devem ser convertidas para tipos adequados.



In [ ]:
sensores["data_hora"] = pd.to_datetime(sensores["data_hora"])
manutencao["ultima_manutencao"] = pd.to_datetime(manutencao["ultima_manutencao"])

sensores.dtypes

## Padronizando textos

Também podemos remover espaços desnecessários e padronizar textos.



In [ ]:
cadastro["fabricante"] = cadastro["fabricante"].str.strip()
cadastro["linha"] = cadastro["linha"].str.strip()

cadastro

# 🔗 6. TRANSFORM — Combinando dados

Agora temos três DataFrames.

Precisamos relacioná-los.

A coluna `motor` funciona como uma **chave** para fazer a combinação.

Podemos utilizar `merge()`.


In [ ]:
dados_etl = sensores.merge(
    cadastro,
    on="motor",
    how="left"
)

dados_etl

Agora vamos incorporar os dados de manutenção.


In [ ]:
dados_etl = dados_etl.merge(
    manutencao,
    on="motor",
    how="left"
)

dados_etl

Observe que agora temos uma única tabela com informações provenientes das três fontes.

Esse é um dos objetivos mais importantes de um processo ETL:

> **Transformar diferentes fontes em uma estrutura integrada para análise.**


# 🧮 7. Criando novas informações

Durante a transformação podemos criar novas variáveis.

Por exemplo, podemos calcular a diferença entre a potência medida e a potência nominal.

Para isso, primeiro vamos criar uma potência medida simplificada para fins didáticos.


In [ ]:
dados_etl["potencia_medida"] = (
    dados_etl["corrente"] * dados_etl["tensao"] / 1000
)

dados_etl[[
    "motor",
    "corrente",
    "tensao",
    "potencia_medida",
    "potencia_nominal"
]]

Agora podemos criar uma variável de diferença.



In [ ]:
dados_etl["diferenca_potencia"] = (
    dados_etl["potencia_medida"] - dados_etl["potencia_nominal"]
)

dados_etl[[
    "motor",
    "potencia_medida",
    "potencia_nominal",
    "diferenca_potencia"
]]

# 🧹 8. Transformação e qualidade

Vamos verificar se a base integrada possui problemas.



In [ ]:
dados_etl.info()

In [ ]:
dados_etl.isnull().sum()

In [ ]:
dados_etl.duplicated().sum()

Mesmo depois da integração, devemos executar verificações de qualidade.

ETL não é apenas "juntar tabelas".

Ele também envolve preparar os dados para que possam ser utilizados com segurança.


# 💾 9. LOAD — Carregando os dados

Depois da transformação, podemos salvar a base.

Uma opção simples é CSV.


In [ ]:
arquivo_saida = "motores_etl.csv"

dados_etl.to_csv(arquivo_saida, index=False)

print("Arquivo criado:", arquivo_saida)

Podemos conferir o arquivo carregando-o novamente.


In [ ]:
dados_carregados = pd.read_csv(arquivo_saida)

dados_carregados.head()

Também podemos verificar o tamanho da base carregada.


In [ ]:
dados_carregados.shape

# 🔄 10. Construindo um fluxo ETL

Agora vamos organizar o processo em funções.

Isso deixa o código mais reutilizável.



In [ ]:
def extrair_dados():
    sensores = pd.DataFrame({
        "motor": ["M001", "M001", "M002", "M002", "M003", "M003"],
        "data_hora": [
            "2026-02-01 08:00", "2026-02-01 09:00",
            "2026-02-01 08:00", "2026-02-01 09:00",
            "2026-02-01 08:00", "2026-02-01 09:00"
        ],
        "corrente": [12.4, 12.8, 10.8, 11.0, 14.1, 14.3],
        "tensao": [380, 379, 380, 381, 382, 381],
        "vibracao": [1.8, 2.1, 1.2, 1.3, 2.4, 2.6],
        "temperatura": [62.3, 64.1, 55.1, 56.0, 67.2, 68.4],
        "rpm": [1750, 1748, 1752, 1750, 1740, 1738]
    })

    cadastro = pd.DataFrame({
        "motor": ["M001", "M002", "M003"],
        "modelo": ["MX-100", "MX-100", "MX-200"],
        "fabricante": ["MotorTech", "MotorTech", "PowerMotor"],
        "potencia_nominal": [7.5, 6.5, 9.0],
        "linha": ["Linha A", "Linha A", "Linha B"]
    })

    manutencao = pd.DataFrame({
        "motor": ["M001", "M002", "M003"],
        "ultima_manutencao": ["2026-01-15", "2026-01-20", "2025-12-18"],
        "horas_operacao": [1254, 824, 2104],
        "status_manutencao": ["Em dia", "Em dia", "Atenção"]
    })

    return sensores, cadastro, manutencao

In [ ]:
def transformar_dados(sensores, cadastro, manutencao):
    sensores["data_hora"] = pd.to_datetime(sensores["data_hora"])
    manutencao["ultima_manutencao"] = pd.to_datetime(
        manutencao["ultima_manutencao"]
    )

    cadastro["fabricante"] = cadastro["fabricante"].str.strip()
    cadastro["linha"] = cadastro["linha"].str.strip()

    dados = sensores.merge(cadastro, on="motor", how="left")
    dados = dados.merge(manutencao, on="motor", how="left")

    dados["potencia_medida"] = (
        dados["corrente"] * dados["tensao"] / 1000
    )

    dados["diferenca_potencia"] = (
        dados["potencia_medida"] - dados["potencia_nominal"]
    )

    return dados

In [ ]:
def carregar_dados(dados, arquivo="motores_etl_final.csv"):
    dados.to_csv(arquivo, index=False)
    return arquivo

In [ ]:
sensores, cadastro, manutencao = extrair_dados()

dados_finais = transformar_dados(
    sensores,
    cadastro,
    manutencao
)

arquivo = carregar_dados(dados_finais)

print("Processo ETL concluído!")
print("Arquivo:", arquivo)

dados_finais

# 🧠 11. Entendendo o fluxo

Nosso processo ficou:

```text
EXTRACT
   ↓
sensores
cadastro
manutenção
   ↓
TRANSFORM
   ↓
padronização
merge
novas variáveis
validação
   ↓
LOAD
   ↓
motores_etl_final.csv
```

Em um projeto real, as fontes podem ser muito maiores e mais complexas.

A lógica, porém, continua semelhante.


# 📝 12. Exercícios

## Exercício 1 — Conceito

Explique com suas palavras:

- O que significa Extract?
- O que significa Transform?
- O que significa Load?



In [ ]:
# Sua resposta



## Exercício 2 — Fontes

Liste pelo menos **5 fontes diferentes** que poderiam fornecer dados para um projeto de Mineração de Dados.


In [ ]:
# Sua resposta



## Exercício 3 — Extração

Crie um DataFrame chamado `producao` com pelo menos 5 registros contendo:

- motor;
- produto;
- quantidade_produzida;
- turno.



In [ ]:
# Sua resposta



## Exercício 4 — Transformação

Adicione ao DataFrame `producao` uma coluna chamada `eficiencia`.

Utilize uma fórmula criada por você e explique o que ela representa.


In [ ]:
# Sua resposta



## Exercício 5 — Integração

Utilize `merge()` para combinar `producao` com o cadastro de motores.

O que acontece quando uma chave não existe em uma das tabelas?


In [ ]:
# Sua resposta



## Exercício 6 — Qualidade

Verifique:

- valores ausentes;
- duplicidades;
- tipos;
- valores inconsistentes.

na sua tabela integrada.


In [ ]:
# Sua resposta



## Exercício 7 — Load

Salve sua tabela integrada em um arquivo CSV chamado:

```text
producao_etl.csv
```

Depois carregue o arquivo novamente utilizando Pandas.


In [ ]:
# Sua resposta



## Exercício 8 — Consultando o resultado

Na base `dados_finais`, encontre:

- o motor com maior temperatura;
- o motor com maior vibração;
- o motor com maior número de horas de operação.



In [ ]:
# Sua resposta



## Exercício 9 — Fluxo

Represente com código ou Markdown o fluxo ETL do projeto do motor elétrico.

Exemplo:

```text
Fonte A ──┐
Fonte B ──┼──> Transformação ──> Base final
Fonte C ──┘
```



In [ ]:
# Sua resposta



## Exercício 10 — Função

Crie uma função chamada:

```python
executar_etl()
```

Ela deverá:

1. extrair os dados;
2. transformar os dados;
3. salvar o resultado;
4. informar que o processo terminou.



In [ ]:
# Sua resposta



# 🔎 13. Desafio — ETL de um projeto real

Agora pense no seu projeto de avaliação.

Você deverá identificar:

### 1. Fontes de dados

Onde os dados serão obtidos?

### 2. Extração

Como os dados serão coletados?

### 3. Transformação

Quais tratamentos serão necessários?

### 4. Integração

Será necessário combinar diferentes fontes?

### 5. Carga

Onde os dados preparados serão armazenados?

Monte o seguinte quadro:

| Etapa | Decisão do projeto |
|---|---|
| Extract | ... |
| Transform | ... |
| Integração | ... |
| Load | ... |

> **Importante:** não é necessário implementar o ETL completo do projeto agora. O objetivo é começar a planejar como os dados serão obtidos e preparados.


In [ ]:
# Planejamento do seu ETL



# 📌 14. Checklist da Aula

- [ ] Entendo o conceito de ETL;
- [ ] Sei diferenciar Extract, Transform e Load;
- [ ] Sei criar DataFrames a partir de diferentes fontes simuladas;
- [ ] Sei utilizar `merge()`;
- [ ] Sei transformar tipos;
- [ ] Sei criar novas variáveis;
- [ ] Sei verificar a qualidade após a integração;
- [ ] Sei salvar uma base processada;
- [ ] Consigo organizar um fluxo ETL em funções;
- [ ] Consigo pensar no ETL necessário para meu projeto.

---

# 🎯 Conclusão

Nas últimas aulas construímos uma sequência:

```text
Aula 1 → Conceitos de Mineração de Dados
Aula 2 → KDD e CRISP-DM
Aula 3 → Manipulação com Pandas
Aula 4 → Limpeza e tratamento
Aula 5 → ETL
```

Agora já sabemos **como obter, transformar e preparar dados**.

Na próxima aula vamos trabalhar com uma fonte muito comum de dados:

> 🌐 **Web Scraping — extração de dados de páginas da web.**

A partir daí começaremos a lidar com dados externos ao nosso notebook.
